# 🥭 CS22032 Essentials of AI - Custom Mobile Photos Training Notebook
**Module Code**: CS22032 Essentials of Artificial Intelligence  
**Topic**: AI-Based Intelligent Mango Quality & Ripeness Assessment System  
**Platform**: Google Colab Cloud GPU Acceleration (NVIDIA T4)  

--- 
### 📋 Overview
This notebook allows you to upload **real photos of mangoes taken from your mobile phone** and train a 3-block Convolutional Neural Network (`MangoCNN`) in PyTorch.

#### Quality Classes:
- **`Grade_A_Ripe`**: Fresh / Yellow Ripe Mangoes 🥭
- **`Grade_B_Unripe`**: Green / Immature Mangoes 🍏
- **`Grade_C_Overripe`**: Overripe / Bruised / Defective Mangoes 🍂

### ⚡ Step 1: Verify Google Colab GPU Hardware Acceleration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
import os
import zipfile
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Connected to Google Colab Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Accelerator Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not enabled. Go to Runtime -> Change runtime type -> Select T4 GPU")

### 📁 Step 2: Upload Your Mobile Photos ZIP File
Upload your `my_mango_dataset.zip` file containing subfolders (`Grade_A_Ripe`, `Grade_B_Unripe`, `Grade_C_Overripe`).

In [ ]:
print("📥 Click below to upload 'my_mango_dataset.zip' from your computer:")
uploaded = files.upload()

# Extract uploaded zip file
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print(f"✅ Successfully unzipped {filename} into 'dataset/' folder!")

### 🧠 Step 3: Define PyTorch EfficientNet-B0 Transfer Learning Architecture
EfficientNet-B0 Transfer Learning with 2-Stage Unfreezing Support.

In [ ]:
import torchvision.models as models
import torch
import torch.nn as nn

class MangoEfficientNet(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super(MangoEfficientNet, self).__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.base_model = models.efficientnet_b0(weights=weights)
        
        self.freeze_backbone()
            
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def freeze_backbone(self):
        for param in self.base_model.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self, unfreeze_from_block=4):
        total_blocks = len(self.base_model.features)
        print(f"[INFO] Unfreezing EfficientNet-B0 backbone from block {unfreeze_from_block} to {total_blocks - 1}...")
        for i, block in enumerate(self.base_model.features):
            if i >= unfreeze_from_block:
                for param in block.parameters():
                    param.requires_grad = True

    def get_backbone_params(self):
        return [p for p in self.base_model.features.parameters() if p.requires_grad]

    def get_classifier_params(self):
        return [p for p in self.base_model.classifier.parameters() if p.requires_grad]

    def forward(self, x):
        return self.base_model(x)

CLASS_NAMES = ['Grade_A_Ripe', 'Grade_B_Unripe', 'Grade_C_Overripe', 'Non_Mango']
model = MangoEfficientNet(num_classes=len(CLASS_NAMES)).to(device)
print(f"?? Loaded EfficientNet-B0 Transfer Learning Model (Classes: {len(CLASS_NAMES)})")

### ⚡ Step 4: 2-Stage GPU Fine-Tuning (Stage 1: Frozen -> Stage 2: Unfrozen Differential LR)

In [ ]:
class CustomMangoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for idx, class_name in enumerate(CLASS_NAMES):
            for root, _, files in os.walk(root_dir):
                if class_name in root:
                    for f in files:
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                            self.samples.append((os.path.join(root, f), idx))
                            
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CustomMangoDataset('dataset', transform=transform_train)
if len(dataset) == 0:
    print("⚠️ Warning: Dataset empty! Ensure subfolders match CLASS_NAMES inside dataset/")
else:
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

criterion = FocalLoss(gamma=2.0)
    TOTAL_EPOCHS = 15
    STAGE1_EPOCHS = 5
    optimizer = optim.Adam(model.get_classifier_params(), lr=0.0005, weight_decay=1e-4)

    print(f"🚀 Starting 2-Stage EfficientNet-B0 Fine-Tuning on GPU for {TOTAL_EPOCHS} Epochs...")
    for epoch in range(1, TOTAL_EPOCHS + 1):
        # Transition to Stage 2: Unfreeze Upper Layers & Apply Differential LR
        if epoch == STAGE1_EPOCHS + 1:
            print("\n🔥 STAGE 2: Unfreezing Upper EfficientNet-B0 Backbone for Deep Fine-Tuning!")
            model.unfreeze_backbone(unfreeze_from_block=10)
            optimizer = optim.Adam([
                {'params': model.get_backbone_params(), 'lr': 1e-5},
                {'params': model.get_classifier_params(), 'lr': 1e-4}
            ], weight_decay=1e-4)
            print("⚡ Configured Differential LRs: Backbone = 1e-5, Head = 1e-4\n")

        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outs, 1)
            total += lbls.size(0)
            correct += (preds == lbls).sum().item()
        acc = (correct / total) * 100 if total > 0 else 0
        stage_name = "Stage 1 (Head)" if epoch <= STAGE1_EPOCHS else "Stage 2 (Fine-Tuning)"
        print(f"[{stage_name}] Epoch [{epoch:02d}/{TOTAL_EPOCHS}] - Loss: {running_loss/max(1,total):.4f} - Acc: {acc:.2f}%")

    # Save Model Weights
    torch.save(model.state_dict(), 'mango_model.pth')
    print("\n🎉 Fine-Tuned EfficientNet-B0 Model Saved as 'mango_model.pth'!")


### 📥 Step 5: Download Model File to Local Machine

In [ ]:
files.download('mango_model.pth')
print("📥 Downloading 'mango_model.pth' to your computer...")